# Glitch demo

- build a sequence of 3 glitches with `Glitch`
- plot the waveform
- load to device and force trigger

One bit duration is `1 / (2 * F_CPU)`, use `ns()`, `us()` and `ms()` helpers if needed.

In [ ]:
%matplotlib qt
import matplotlib.pyplot as plt

from glitcher import Glitch, Glitcher
from dma_compiler import compile_dma_sequence
from waveform import plot_waveforms

F_CPU = 250e6

## glitch builder
`push_glitch(cycle, pattern)` outputs `pattern` at `cycle`. Multiple glitches must be in order and must not overlap.

Pattern is output MSB first and consists of one or multiple 32-bit words. `g.pulse(n)` makes `n` high bits at the start of
the word. A raw `0b...` pattern is a 32-bit word, so shift it to the top to start it at `cycle`.

example: `0b1101` will end up with 28 bits of zeros and the actual glitch will be delayed by 28 cycles.

In [ ]:
g = Glitch(F_CPU)
g.push_glitch(g.us(1), g.pulse(10))             # 20 ns pulse at 1 us
g.push_glitch(g.us(1.5) + 16, g.pulse(42))      # 84 ns pulse at 1.5 us + 16 cycles (1.532 us), 2 words wide
g.push_glitch(g.us(6), 0b1010101)               # four 2 ns pulses at 6 us

for delay, pattern in g.seq:
    print(f'delay {delay:5} bits, pattern 0x{pattern:X}')

DMA compiler builds the output sequence

In [ ]:
compiled = compile_dma_sequence(g.seq)
zeros_first, durations, patterns = compiled
print(f'zeros_first={zeros_first}, blocks={durations} words, {len(patterns)} pattern words')

plot the glitch

In [ ]:
ax = plot_waveforms(compiled, freq=F_CPU, channels=0, cycles=True)

## upload and execute

In [ ]:
dev = Glitcher()
dev.set_freq(F_CPU)

for i in range(2048):
    dev.load_glitch(g)
    dev.force_trigger()
    print(dev.get_trigger_state().name)

In [ ]:
# external trigger:
# dev.set_trigger_polarity(True)    # rising edge
# dev.set_trigger_edge(0)           # 1st edge
# dev.arm()
# while dev.get_trigger_state() != Glitcher.State.TRIGGERED:
#     pass

In [ ]:
dev.close()